# Online Retail II — SQL Business Analysis

This notebook uses SQL to analyze the cleaned Online Retail II dataset and answer the business questions defined in the project README.


#### Project Setup
The cleaned Online Retail II dataset is loaded into DuckDB so SQL can be used for downstream business analysis.

In [ ]:
# Install packages
from google.colab import drive
import pandas as pd
import duckdb

# Connect to Google Drive
drive.mount('/content/drive')

In [ ]:
csv_path = "/content/drive/MyDrive/Working/Resume 📃/Portfolio/Retail Revenue Intelligence/Data/online_retail_clean.csv"

con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE VIEW retail AS
    SELECT *
    FROM read_csv_auto('{csv_path}')
""")

In [ ]:
con.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT Invoice) AS invoice_count,
        MIN(InvoiceDate) AS earliest_date,
        MAX(InvoiceDate) AS latest_date
    FROM retail
""").df()

#### Business Questions

**Revenue**

1. How has revenue changed over time?
2. What periods generate the highest and lowest revenue?


**Product**

3. Which products generate the most revenue
4. Which products have the highest sales volume?

**Geographic**

5. Which countries generate the most revenue?
6. How does revenue performance outside the United Kingdom compare across markets?

**Customer**

7. Which identified customers generate the most revenue?
8. How does purchasing behavior vary across identified customers?

#### KPI Definitions

| KPI | Definition |
|---|---|
| Net Revenue | Total `Analytical_Revenue`, including the negative impact of cancellations and returns. |
| Gross Sales | Revenue generated from `Standard Sale` transactions before cancellations and returns. |
| Orders | Distinct invoices classified as standard sales. |
| Units Sold | Total quantity from standard sale transactions. |
| Average Order Value | Gross Sales divided by the number of standard sale orders. |
| Return Value | Absolute revenue value associated with cancellation/return transactions. |
| Return Rate | Return Value as a percentage of Gross Sales. |
| Identified Customers | Distinct non-null `Customer ID` values associated with standard sales. |

#### Analysis

##### Revenue Analysis

**Business Question 1: How has revenue changed over time?**

Analyze monthly net revenue to identify overall trends, growth patterns, seasonality, and periods of unusually high or low performance.



**Solution**

* Monthly net revenue shows a recurring seasonal pattern, with performance increasing substantially from September through November in both 2010 and 2011.
* November was the strongest complete month in each year, reaching approximately 1.42 million in 2010 and 1.46 million in 2011.
* Revenue then declined following the peak period. December 2011 is a partial month containing transactions only through
* December 9 and should not be directly compared with complete months.
* The dataset generated approximately 20.52 million in gross sales and 19.06 million in net revenue after accounting for cancellations and returns.
* Standard-sale orders averaged approximately 512, while return activity represented approximately 1.47 million, or 7.14% of gross sales.

In [ ]:
con.execute("""
SELECT
    SUM(Analytical_Revenue) AS Net_Revenue,

    SUM(
        CASE
            WHEN Transaction_Type = 'Standard Sale'
            THEN Line_Revenue
            ELSE 0
        END
    ) AS Gross_Sales,

    COUNT(
        DISTINCT CASE
            WHEN Transaction_Type = 'Standard Sale'
            THEN Invoice
        END
    ) AS Orders,

    SUM(
        CASE
            WHEN Transaction_Type = 'Standard Sale'
            THEN Quantity
            ELSE 0
        END
    ) AS Units_Sold,

    COUNT(
        DISTINCT CASE
            WHEN Transaction_Type = 'Standard Sale'
                 AND "Customer ID" IS NOT NULL
            THEN "Customer ID"
        END
    ) AS Identified_Customers
FROM retail
""").df()

,Net_Revenue,Gross_Sales,Orders,Units_Sold,Identified_Customers
0,1.905738e+07,2.052305e+07,40077,11239346.0,5878


In [ ]:
con.execute("""
SELECT
    SUM(CASE
        WHEN Transaction_Type = 'Standard Sale'
        THEN Line_Revenue
        ELSE 0
    END) /
    COUNT(DISTINCT CASE
        WHEN Transaction_Type = 'Standard Sale'
        THEN Invoice
    END) AS Average_Order_Value,

    ABS(SUM(CASE
        WHEN Transaction_Type = 'Cancellation/Return'
        THEN Line_Revenue
        ELSE 0
    END)) AS Return_Value,

    ABS(SUM(CASE
        WHEN Transaction_Type = 'Cancellation/Return'
        THEN Line_Revenue
        ELSE 0
    END))
    /
    SUM(CASE
        WHEN Transaction_Type = 'Standard Sale'
        THEN Line_Revenue
        ELSE 0
    END) * 100 AS Return_Rate
FROM retail
""").df()

,Average_Order_Value,Return_Value,Return_Rate
0,512.090561,1465677.23,7.141614


In [ ]:
monthly_revenue = con.execute("""
SELECT
    DATE_TRUNC('month', InvoiceDate) AS Revenue_Month,
    SUM(Analytical_Revenue) AS Net_Revenue
FROM retail
GROUP BY Revenue_Month
ORDER BY Revenue_Month
""").df()

monthly_revenue

,Revenue_Month,Net_Revenue
0,2009-12-01,799847.110
1,2010-01-01,624032.892
2,2010-02-01,533091.426
3,2010-03-01,765848.761
4,2010-04-01,644174.792
5,2010-05-01,615322.830
6,2010-06-01,679786.610
7,2010-07-01,619268.150
8,2010-08-01,656776.340
9,2010-09-01,853650.431


**Business Question 2: What periods generate the highest and lowest revenue?**

Identify the strongest and weakest revenue periods to better understand seasonal performance and fluctuations in business activity.

**Solution**

* November 2011 generated the highest monthly net revenue at approximately 1.46 million, followed closely by November 2010 at approximately 1.42 million.

* Four of the five highest-revenue months occurred during September through November, reinforcing the strong fall seasonal pattern observed in the revenue trend analysis.

* Among complete months, April 2011 generated the lowest net revenue at approximately 493,000, followed by February 2011 at approximately 498,000.

* December 2011 produced the lowest reported monthly revenue at approximately 434,000, but it contains transactions only through December 9 and should not be treated as the lowest-performing complete month.

In [ ]:
highest_revenue_months = con.execute("""
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', InvoiceDate) AS Revenue_Month,
        SUM(Analytical_Revenue) AS Net_Revenue
    FROM retail
    GROUP BY Revenue_Month
)

SELECT
    Revenue_Month,
    Net_Revenue
FROM monthly_revenue
ORDER BY Net_Revenue DESC
LIMIT 5
""").df()

highest_revenue_months

,Revenue_Month,Net_Revenue
0,2011-11-01,1461756.250
1,2010-11-01,1422654.642
2,2010-10-01,1084094.220
3,2011-10-01,1070704.670
4,2011-09-01,1019687.622


In [ ]:
lowest_revenue_months = con.execute("""
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', InvoiceDate) AS Revenue_Month,
        SUM(Analytical_Revenue) AS Net_Revenue
    FROM retail
    GROUP BY Revenue_Month
)

SELECT
    Revenue_Month,
    Net_Revenue
FROM monthly_revenue
ORDER BY Net_Revenue ASC
LIMIT 5
""").df()

lowest_revenue_months

,Revenue_Month,Net_Revenue
0,2011-12-01,433686.010
1,2011-04-01,493207.121
2,2011-02-01,498062.650
3,2010-02-01,533091.426
4,2011-01-01,560000.260


##### Product Analysis

**Business Question 3: Which products generate the most revenue?**

Analyze product-level net revenue to identify the products that contribute the most revenue to the business.


**Solution**

* REGENCY CAKESTAND 3 TIER generated the highest net revenue at approximately 314,500.

* WHITE HANGING HEART T-LIGHT HOLDER ranked second at approximately 248,700, followed by JUMBO BAG RED RETROSPOT at approximately 178,700.

* Revenue was concentrated among the highest-performing products, with the top two products generating substantially more revenue than the remaining products in the top 10.

In [ ]:
# MODE() gives us the most commonly used product description among standard sales for each StockCode
top_revenue_products = con.execute("""
SELECT
    StockCode,
    MODE(Description) FILTER (
        WHERE Transaction_Type = 'Standard Sale'
    ) AS Description,
    SUM(Analytical_Revenue) AS Net_Revenue
FROM retail
WHERE Activity_Type = 'Merchandise'
GROUP BY StockCode
ORDER BY Net_Revenue DESC
LIMIT 10
""").df()

top_revenue_products

,StockCode,Description,Net_Revenue
0,22423,REGENCY CAKESTAND 3 TIER,314513.47
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,248678.81
2,85099B,JUMBO BAG RED RETROSPOT,178719.21
3,47566,PARTY BUNTING,147157.43
4,84879,ASSORTED COLOUR BIRD ORNAMENT,128907.01
5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,116422.49
6,79321,CHILLI LIGHTS,80237.48
7,22197,SMALL POPCORN HOLDER,78935.92
8,22386,JUMBO BAG PINK POLKADOT,75518.22
9,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,71031.15


**Business Question 4: Which products have the highest sales volume?**

Analyze product-level units sold to identify the products with the highest sales volume.

**Solution**

* WORLD WAR 2 GLIDERS ASSTD DESIGNS had the highest sales volume at approximately 106,400 units sold.

* JUMBO BAG RED RETROSPOT ranked second at approximately 96,900 units, followed by PACK OF 72 RETROSPOT CAKE CASES at approximately 95,000 units.

* Several high-volume products also appeared among the top revenue-generating products, including JUMBO BAG RED RETROSPOT, WHITE HANGING HEART T-LIGHT HOLDER, SMALL POPCORN HOLDER, and ASSORTED COLOUR BIRD ORNAMENT.

* The highest-volume product was not the highest-revenue product, showing that unit volume and revenue contribution are not necessarily the same.

In [ ]:
top_volume_products = con.execute("""
SELECT
    StockCode,
    MODE(Description) FILTER (
        WHERE Transaction_Type = 'Standard Sale'
    ) AS Description,
    SUM(
        CASE
            WHEN Transaction_Type = 'Standard Sale'
            THEN Quantity
            ELSE 0
        END
    ) AS Units_Sold
FROM retail
WHERE Activity_Type = 'Merchandise'
GROUP BY StockCode
ORDER BY Units_Sold DESC
LIMIT 10
""").df()

top_volume_products

,StockCode,Description,Units_Sold
0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,106379.0
1,85099B,JUMBO BAG RED RETROSPOT,96931.0
2,21212,PACK OF 72 RETROSPOT CAKE CASES,94972.0
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,94323.0
4,22197,SMALL POPCORN HOLDER,88537.0
5,23843,"PAPER CRAFT , LITTLE BIRDIE",80995.0
6,84879,ASSORTED COLOUR BIRD ORNAMENT,80293.0
7,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033.0
8,17003,BROCADE RING PURSE,70405.0
9,21977,PACK OF 60 PINK PAISLEY CAKE CASES,56230.0


##### Geographic Analysis

**Business Question 5: Which countries generate the most revenue?**

Analyze net revenue by country to identify the geographic markets contributing the most revenue to the business.

**Solution**

* The United Kingdom generated the most net revenue by a substantial margin at approximately 16.19 million.

* EIRE ranked second at approximately 610,000, followed by the Netherlands at approximately 548,000.

* Germany and France also represented major non-UK markets, generating approximately 413,000 and 322,000 in net revenue respectively.

* Revenue was heavily concentrated in the United Kingdom, while international revenue was distributed across a much smaller set of markets.

In [ ]:
top_revenue_countries = con.execute("""
SELECT
    Country,
    SUM(Analytical_Revenue) AS Net_Revenue
FROM retail
GROUP BY Country
ORDER BY Net_Revenue DESC
LIMIT 10
""").df()

top_revenue_countries

,Country,Net_Revenue
0,United Kingdom,1.618622e+07
1,EIRE,6.102443e+05
2,Netherlands,5.483324e+05
3,Germany,4.125178e+05
4,France,3.219285e+05
5,Australia,1.665119e+05
6,Switzerland,9.942536e+04
7,Spain,9.106476e+04
8,Sweden,8.780942e+04
9,Denmark,6.445959e+04


**Business Question 6: How does revenue performance outside the United Kingdom compare across markets?**

Analyze net revenue across non-UK markets to identify the strongest international markets and better understand geographic revenue performance outside the dominant UK market.

**Solution**

* EIRE generated the highest net revenue outside the United Kingdom at approximately 610,000, supported by 626 orders and an average order value of approximately 1,053.

* The Netherlands generated approximately 548,000 from only 228 orders but had the highest average order value among the leading international markets at approximately 2,430.

* Germany generated approximately 413,000 across 789 orders, the highest order count among the top international markets, but had a substantially lower average order value of approximately 539.

* International markets therefore show different revenue patterns, with some markets driven by higher order volume and others by larger average order values.

In [ ]:
international_revenue = con.execute("""
SELECT
    Country,
    SUM(Analytical_Revenue) AS Net_Revenue
FROM retail
WHERE Country <> 'United Kingdom'
GROUP BY Country
ORDER BY Net_Revenue DESC
LIMIT 10
""").df()

international_revenue

,Country,Net_Revenue
0,EIRE,610244.270
1,Netherlands,548332.350
2,Germany,412517.831
3,France,321928.530
4,Australia,166511.920
5,Switzerland,99425.360
6,Spain,91064.760
7,Sweden,87809.420
8,Denmark,64459.590
9,Belgium,63228.390


In [ ]:
international_performance = con.execute("""
SELECT
    Country,

    SUM(Analytical_Revenue) AS Net_Revenue,

    COUNT(
        DISTINCT CASE
            WHEN Transaction_Type = 'Standard Sale'
            THEN Invoice
        END
    ) AS Orders,

    SUM(
        CASE
            WHEN Transaction_Type = 'Standard Sale'
            THEN Line_Revenue
            ELSE 0
        END
    )
    /
    COUNT(
        DISTINCT CASE
            WHEN Transaction_Type = 'Standard Sale'
            THEN Invoice
        END
    ) AS Average_Order_Value

FROM retail

WHERE Country <> 'United Kingdom'

GROUP BY Country
ORDER BY Net_Revenue DESC
LIMIT 10
""").df()

international_performance

,Country,Net_Revenue,Orders,Average_Order_Value
0,EIRE,610244.270,626,1052.953674
1,Netherlands,548332.350,228,2429.998860
2,Germany,412517.831,789,539.389583
3,France,321928.530,622,563.752701
4,Australia,166511.920,95,1782.641684
5,Switzerland,99425.360,93,1082.880538
6,Spain,91064.760,154,703.790974
7,Sweden,87809.420,105,875.273524
8,Denmark,64459.590,43,1594.899767
9,Belgium,63228.390,149,438.975302


##### Customer Analysis

**Business Question 7: Which identified customers generate the most revenue?**

Analyze customer-level net revenue to identify the identified customers contributing the most revenue to the business.

**Solution**

* Customer 18102 generated the highest net revenue at approximately 570,400.

* Customer 14646 ranked second at approximately 523,300, followed by Customer 14156 at approximately 296,200.

* The top two customers generated substantially more net revenue than the rest of the top 10, indicating that a relatively small number of identified customers contributed a large amount of revenue.

* Customer-level analysis is based only on transactions with a populated `Customer ID`, so unidentified customers are excluded from this ranking.

In [ ]:
top_customers = con.execute("""
SELECT
    "Customer ID",
    SUM(Analytical_Revenue) AS Net_Revenue
FROM retail
WHERE "Customer ID" IS NOT NULL
GROUP BY "Customer ID"
ORDER BY Net_Revenue DESC
LIMIT 10
""").df()

top_customers

,Customer ID,Net_Revenue
0,18102.0,570380.61
1,14646.0,523342.07
2,14156.0,296249.99
3,14911.0,265836.95
4,17450.0,231550.55
5,13694.0,189983.40
6,17511.0,168491.62
7,12415.0,143269.29
8,16684.0,141502.25
9,15061.0,124961.98


**Business Question 8: How does purchasing behavior vary across identified customers?**

Analyze order frequency, purchasing volume, net revenue, and average order value across identified customers to understand differences in customer purchasing behavior.

**Solution**

* The highest-value customers reached similar revenue levels through different purchasing behaviors, demonstrating that revenue contribution is driven by both order frequency and order size.

* Customer 14911 placed the most orders among the top customers with 398 orders, but had a relatively lower average order value of approximately 733.

* In contrast, Customer 12415 placed only 28 orders but had the highest average order value among the top customers at approximately 5,159.

* Customers 18102 and 14646 combined relatively high order frequency with large average order values, resulting in the two highest net revenue totals among identified customers.

In [ ]:
customer_behavior = con.execute("""
SELECT
    "Customer ID",

    COUNT(
        DISTINCT CASE
            WHEN Transaction_Type = 'Standard Sale'
            THEN Invoice
        END
    ) AS Orders,

    SUM(
        CASE
            WHEN Transaction_Type = 'Standard Sale'
            THEN Quantity
            ELSE 0
        END
    ) AS Units_Sold,

    SUM(Analytical_Revenue) AS Net_Revenue,

    SUM(
        CASE
            WHEN Transaction_Type = 'Standard Sale'
            THEN Line_Revenue
            ELSE 0
        END
    )
    /
    COUNT(
        DISTINCT CASE
            WHEN Transaction_Type = 'Standard Sale'
            THEN Invoice
        END
    ) AS Average_Order_Value

FROM retail
WHERE "Customer ID" IS NOT NULL
GROUP BY "Customer ID"
HAVING COUNT(
    DISTINCT CASE
        WHEN Transaction_Type = 'Standard Sale'
        THEN Invoice
    END
) > 0
ORDER BY Net_Revenue DESC
LIMIT 10
""").df()

customer_behavior

,Customer ID,Orders,Units_Sold,Net_Revenue,Average_Order_Value
0,18102.0,145,181645.0,570380.61,4006.807172
1,14646.0,151,367193.0,523342.07,3500.678940
2,14156.0,156,164444.0,296249.99,2010.411346
3,14911.0,398,148010.0,265836.95,732.565452
4,17450.0,51,83934.0,231550.55,4802.828431
5,13694.0,143,188201.0,189983.40,1368.116713
6,17511.0,60,117174.0,168491.62,2868.881167
7,12415.0,28,91447.0,143269.29,5159.227500
8,16684.0,55,104810.0,141502.25,2675.323091
9,15061.0,127,74280.0,124961.98,995.189134
